# Chapter 3 — Kernighan-Lin Algorithm

This chapter covers two iterative graph partitioning algorithms studied in the thesis:
- **Kernighan-Lin** — classic pairwise node-swap heuristic (Section 3.1)
- **Degree-normalised variant** — our own modification (Section 3.2)
- **Comparison** of all three methods on the same reference graphs (Section 3.3)

---

In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd
from scipy.linalg import eigh
from numpy import sqrt

## 3.1 Kernighan-Lin Algorithm

Given a graph $G = (V, E)$ with $|V| = 2n$, start from an arbitrary balanced partition $A \cup B = V$, $|A| = |B| = n$.

For each node $v$ define the **cost difference**:
$$D(v) = E(v) - I(v)$$
where $E(v)$ is the sum of edge weights from $v$ to the **opposite** set and $I(v)$ to the **same** set.

The **gain** from swapping pair $(a, b)$, $a \in A$, $b \in B$:
$$g(a, b) = D(a) + D(b) - 2\,c_{ab}$$

**One pass of the algorithm:**
1. Find pair $(a_1, b_1)$ with maximum gain $g_1$, lock them out
2. Update $D$ values for remaining unlocked nodes
3. Repeat for $(a_2, b_2), \ldots, (a_n, b_n)$, recording $g_1, \ldots, g_n$
4. Find $k = \arg\max \sum_{i=1}^k g_i$ — apply first $k$ swaps if $G_k > 0$
5. Repeat from new partition until no improvement

The algorithm has complexity $\mathcal{O}(n^2 \log n)$ per pass (Algorithm 4 in thesis).

In [ ]:
# ---- Cut size helper ----
def compute_cut_size(G, A, B):
    return sum(G[u][v]['weight'] for u, v in G.edges()
               if (u in A and v in B) or (v in A and u in B))

# ---- Kernighan-Lin algorithm ----
def kernighan_lin_algorithm(G, pos):
    nodes = list(G.nodes)
    half = len(nodes) // 2
    A = set(nodes[:half])
    B = set(nodes[half:])

    colors = ['red' if n in A else 'blue' for n in G.nodes()]
    nx.draw(G, pos, with_labels=True, node_color=colors, edge_color='gray')
    plt.title('Original Graph')
    plt.show()

    iteration = 0
    history = []

    while True:
        D = {}
        for n in G.nodes:
            internal = sum(G[n][nbr]['weight'] for nbr in G.neighbors(n) if nbr in (A if n in A else B))
            external = sum(G[n][nbr]['weight'] for nbr in G.neighbors(n) if nbr in (B if n in A else A))
            D[n] = external - internal

        locked = set()
        gains = []
        swaps = []

        for _ in range(half):
            max_g = float('-inf')
            best_a, best_b = None, None
            for a in A - locked:
                for b in B - locked:
                    c_ab = G[a][b]['weight'] if G.has_edge(a, b) else 0
                    g = D[a] + D[b] - 2 * c_ab
                    if g > max_g:
                        max_g = g
                        best_a, best_b = a, b
            if best_a is None or best_b is None:
                break
            gains.append(max_g)
            swaps.append((best_a, best_b))
            locked.update([best_a, best_b])

            for n in (A | B) - locked:
                if n in A:
                    internal = sum(G[n][nbr]['weight'] for nbr in G.neighbors(n) if nbr in (A - locked))
                    external = sum(G[n][nbr]['weight'] for nbr in G.neighbors(n) if nbr in (B - locked))
                else:
                    internal = sum(G[n][nbr]['weight'] for nbr in G.neighbors(n) if nbr in (B - locked))
                    external = sum(G[n][nbr]['weight'] for nbr in G.neighbors(n) if nbr in (A - locked))
                D[n] = external - internal

        g_sums = np.cumsum(gains)
        k = np.argmax(g_sums) + 1
        max_gain = g_sums[k - 1]

        cut = compute_cut_size(G, A, B)
        history.append({'iteration': iteration + 1, 'cut_weight': cut, 'total_gain': max_gain})
        iteration += 1

        if max_gain <= 0:
            break

        for i in range(k):
            a, b = swaps[i]
            A.remove(a); B.remove(b)
            A.add(b); B.add(a)

    return A, B, history

### Example 3.1 — N=10, M=15

Initial partition: $A = \{0,4,6,7,8\}$, $B = \{1,2,3,5,9\}$ (sequential split).

In [ ]:
# ---- Graph N=10, M=15 ----
N1 = 10
M1 = 15
G1 = nx.gnm_random_graph(N1, M1, seed=42)
pos1 = nx.spring_layout(G1, seed=42)

for u, v in G1.edges():
    G1[u][v]['weight'] = np.round(np.random.uniform(1, 1.0), 2)

A_final1, B_final1, history1 = kernighan_lin_algorithm(G1, pos1)

# ---- Final partition ----
colors_final1 = ['red' if n in A_final1 else 'blue' for n in G1.nodes()]
nx.draw(G1, pos1, with_labels=True, node_color=colors_final1, edge_color='gray')
plt.title('Final Partition — Kernighan-Lin')
plt.show()

df1 = pd.DataFrame(history1)
print('\nAnalysis of iterations:')
print(df1.to_string(index=False))

### Example 3.2 — N=22, M=35

Initial partition: $A = \{1,2,3,5,6,8,9,11,13,14,18\}$, $B = \{0,4,7,10,12,15,16,17,19,20,21\}$.

In [ ]:
# ---- Graph N=22, M=35 ----
N2 = 22
M2 = 35
G2 = nx.gnm_random_graph(N2, M2, seed=30)
pos2 = nx.spring_layout(G2, seed=42)

for u, v in G2.edges():
    G2[u][v]['weight'] = np.round(np.random.uniform(1, 1.0), 2)

A_final2, B_final2, history2 = kernighan_lin_algorithm(G2, pos2)

# ---- Final partition ----
colors_final2 = ['red' if n in A_final2 else 'blue' for n in G2.nodes()]
nx.draw(G2, pos2, with_labels=True, node_color=colors_final2, edge_color='gray')
plt.title('Final Partition — Kernighan-Lin')
plt.show()

df2 = pd.DataFrame(history2)
print('\nAnalysis of iterations:')
print(df2.to_string(index=False))

## 3.2 A Variation of the Algorithm

Our variant normalises the gain by node degree, making it scale-invariant.
For each $c_i \in C$ and $b_j \in B$:

$$d_1(i) = \frac{\sum_{j \in B} A_{ij} - \sum_{\lambda \in C} A_{i\lambda}}{0.001 + \deg(i)}$$

$$d_2(j) = \frac{\sum_{i \in C} A_{ji} - \sum_{\lambda \in B} A_{j\lambda}}{0.001 + \deg(j)}$$

$$dd(i,j) = d_1(i) + d_2(j) - \frac{2A_{ij}}{0.001 + \deg(i)} - \frac{2A_{ji}}{0.001 + \deg(j)}$$

At each iteration swap the pair $(i_0, j_0)$ with maximum $dd$. Stop when $\max_{i,j} dd(i,j) \leq 0$ or cut cost reaches $0$ (Algorithm 5 in thesis).

In [ ]:
# ---- Cut cost helper ----
def compute_cut_cost(G, C, B):
    return sum(G[u][v]['weight'] for u in C for v in B if G.has_edge(u, v))

# ---- Degree-normalised variant ----
def kl_variant(G, pos):
    nodes = list(G.nodes())
    half = len(nodes) // 2
    C = set(nodes[:half])
    B = set(nodes[half:])

    A = nx.to_numpy_array(G, weight='weight')
    history = []

    e = compute_cut_cost(G, C, B)
    e_prev = float('inf')

    color_map = ['red' if node in C else 'blue' for node in G.nodes()]
    nx.draw(G, pos, with_labels=True, node_color=color_map, edge_color='gray')
    plt.title('Original Graph')
    plt.show()

    iteration = 0
    while e_prev - e > 0:
        iteration += 1
        e_prev = e
        max_gain = float('-inf')
        best_pair = None

        for ci in list(C):
            for bj in list(B):
                deg_ci = np.sum(A[ci])
                deg_bj = np.sum(A[bj])
                d1 = (sum(A[ci][list(B)]) - sum(A[ci][list(C)])) / (0.001 + deg_ci)
                d2 = (sum(A[bj][list(C)]) - sum(A[bj][list(B)])) / (0.001 + deg_bj)
                dd = d1 + d2 - (2 * A[ci][bj]) / (0.001 + deg_ci) - (2 * A[bj][ci]) / (0.001 + deg_bj)

                if dd > max_gain:
                    max_gain = dd
                    best_pair = (ci, bj)

        history.append({
            'iteration': iteration,
            'cut_cost': round(e, 2),
            'max_gain': round(max_gain, 2),
        })

        if max_gain <= 0 or e == 0:
            break

        ci, bj = best_pair
        C.remove(ci); C.add(bj)
        B.remove(bj); B.add(ci)
        e = compute_cut_cost(G, C, B)

    final_color_map = ['red' if node in C else 'blue' for node in G.nodes()]
    nx.draw(G, pos, with_labels=True, node_color=final_color_map, edge_color='gray')
    plt.title('Final Graph')
    plt.show()

    return C, B, history

### Example 3.3 — N=10, M=15

Initial partition: $C = \{1,2,3,6,7\}$, $B = \{0,4,5,8,9\}$.

In [ ]:
# ---- Variant on N=10, M=15 ----
G3 = nx.gnm_random_graph(10, 15, seed=42)
pos3 = nx.spring_layout(G3, seed=42)

for u, v in G3.edges():
    G3[u][v]['weight'] = np.round(np.random.uniform(1, 1), 2)

C_final3, B_final3, history3 = kl_variant(G3, pos3)

df3 = pd.DataFrame(history3)
print('\nAnalysis of iterations:')
print(df3.to_string(index=False))

### Example 3.4 — N=22, M=35

Initial partition: $C = \{0,1,2,3,4,5,6,7,8,9,10\}$, $B = \{11,12,13,14,15,16,17,18,19,20,21\}$.

In [ ]:
# ---- Variant on N=22, M=35 ----
N4 = 22
M4 = 35
G4 = nx.gnm_random_graph(n=N4, m=M4, seed=30)
pos4 = nx.spring_layout(G4, seed=42)

for u, v in G4.edges():
    G4[u][v]['weight'] = np.round(np.random.uniform(1, 1), 2)

C_final4, B_final4, history4 = kl_variant(G4, pos4)

df4 = pd.DataFrame(history4)
print('\nAnalysis of iterations:')
print(df4.to_string(index=False))

## 3.3 Comparison Between the Algorithms

We run all three methods — Normalized Cut, Kernighan-Lin, and the normalised variant — on the same reference graphs and compare the cut size directly.

Reference graphs match the thesis (Section 3.3): N=8 M=14, N=12 M=21, N=21 M=35.

In [ ]:
# ---- Normalized Cut helper (for comparison) ----
def normalized_cut_with_metrics(G):
    W = nx.to_numpy_array(G, weight='weight')
    degrees = W.sum(axis=1)
    D = np.diag(degrees)
    D_inv_sqrt = np.diag(1.0 / sqrt(degrees))
    L = D - W
    L_sym = np.eye(len(W)) - D_inv_sqrt @ W @ D_inv_sqrt
    eigvals, eigvecs = eigh(L_sym)
    z1 = eigvecs[:, 1]
    y1 = D_inv_sqrt @ z1
    partition = y1 >= 0
    set_A = np.where(partition)[0]
    set_B = np.where(~partition)[0]
    assoc_A_V = np.sum(W[set_A, :])
    assoc_B_V = np.sum(W[set_B, :])
    cut_AB = np.sum(W[np.ix_(set_A, set_B)])
    ncut = (cut_AB / assoc_A_V) + (cut_AB / assoc_B_V)
    return set_A, set_B, z1, y1, assoc_A_V, assoc_B_V, cut_AB, ncut

### Reference graph: N=8, M=14, seed=42

In [ ]:
# ---- Reference graph 1: N=8, M=14 ----
Gref1 = nx.gnm_random_graph(8, 14, seed=42)
posref1 = nx.spring_layout(Gref1, seed=42)
for u, v in Gref1.edges():
    Gref1[u][v]['weight'] = np.round(np.random.uniform(1, 1.0), 2)

nc_A1, nc_B1, *_ = normalized_cut_with_metrics(Gref1)
nc_A1, nc_B1 = set(nc_A1.tolist()), set(nc_B1.tolist())

kl_A1, kl_B1, kl_h1 = kernighan_lin_algorithm(Gref1, posref1)

# ---- Variant (suppress plot) ----
import io, contextlib
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    plt.ioff()
var_C1, var_B1, var_h1 = kl_variant(Gref1, posref1)
plt.ion()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, A_set, B_set, title in [
    (axes[0], nc_A1, nc_B1, f'Normalized Cut (cut={compute_cut_size(Gref1, nc_A1, nc_B1):.0f})'),
    (axes[1], kl_A1, kl_B1, f'Kernighan-Lin (cut={compute_cut_size(Gref1, kl_A1, kl_B1):.0f})'),
    (axes[2], var_C1, var_B1, f'KL Variant (cut={compute_cut_size(Gref1, var_C1, var_B1):.0f})'),
]:
    colors = ['red' if n in A_set else 'blue' for n in Gref1.nodes()]
    nx.draw(Gref1, posref1, with_labels=True, node_color=colors, edge_color='gray',
            node_size=500, font_color='white', font_weight='bold', ax=ax)
    ax.set_title(title, fontsize=10)
plt.suptitle('Reference Graph N=8, M=14, seed=42', fontsize=12)
plt.tight_layout()
plt.show()

print(pd.DataFrame(kl_h1).to_string(index=False))

### Reference graph: N=12, M=21, seed=42

In [ ]:
# ---- Reference graph 2: N=12, M=21 ----
Gref2 = nx.gnm_random_graph(12, 21, seed=42)
posref2 = nx.spring_layout(Gref2, seed=42)
for u, v in Gref2.edges():
    Gref2[u][v]['weight'] = np.round(np.random.uniform(1, 1.0), 2)

nc_A2, nc_B2, *_ = normalized_cut_with_metrics(Gref2)
nc_A2, nc_B2 = set(nc_A2.tolist()), set(nc_B2.tolist())
kl_A2, kl_B2, kl_h2 = kernighan_lin_algorithm(Gref2, posref2)

buf2 = io.StringIO()
with contextlib.redirect_stdout(buf2):
    plt.ioff()
var_C2, var_B2, var_h2 = kl_variant(Gref2, posref2)
plt.ion()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, A_set, B_set, title in [
    (axes[0], nc_A2, nc_B2, f'Normalized Cut (cut={compute_cut_size(Gref2, nc_A2, nc_B2):.0f})'),
    (axes[1], kl_A2, kl_B2, f'Kernighan-Lin (cut={compute_cut_size(Gref2, kl_A2, kl_B2):.0f})'),
    (axes[2], var_C2, var_B2, f'KL Variant (cut={compute_cut_size(Gref2, var_C2, var_B2):.0f})'),
]:
    colors = ['red' if n in A_set else 'blue' for n in Gref2.nodes()]
    nx.draw(Gref2, posref2, with_labels=True, node_color=colors, edge_color='gray',
            node_size=500, font_color='white', font_weight='bold', ax=ax)
    ax.set_title(title, fontsize=10)
plt.suptitle('Reference Graph N=12, M=21, seed=42', fontsize=12)
plt.tight_layout()
plt.show()

print(pd.DataFrame(kl_h2).to_string(index=False))

### Reference graph: N=21, M=35, seed=30

In [ ]:
# ---- Reference graph 3: N=21, M=35 ----
Gref3 = nx.gnm_random_graph(21, 35, seed=30)
posref3 = nx.spring_layout(Gref3, seed=42)
for u, v in Gref3.edges():
    Gref3[u][v]['weight'] = np.round(np.random.uniform(1, 1.0), 2)

nc_A3, nc_B3, *_ = normalized_cut_with_metrics(Gref3)
nc_A3, nc_B3 = set(nc_A3.tolist()), set(nc_B3.tolist())
kl_A3, kl_B3, kl_h3 = kernighan_lin_algorithm(Gref3, posref3)

buf3 = io.StringIO()
with contextlib.redirect_stdout(buf3):
    plt.ioff()
var_C3, var_B3, var_h3 = kl_variant(Gref3, posref3)
plt.ion()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, A_set, B_set, title in [
    (axes[0], nc_A3, nc_B3, f'Normalized Cut (cut={compute_cut_size(Gref3, nc_A3, nc_B3):.0f})'),
    (axes[1], kl_A3, kl_B3, f'Kernighan-Lin (cut={compute_cut_size(Gref3, kl_A3, kl_B3):.0f})'),
    (axes[2], var_C3, var_B3, f'KL Variant (cut={compute_cut_size(Gref3, var_C3, var_B3):.0f})'),
]:
    colors = ['red' if n in A_set else 'blue' for n in Gref3.nodes()]
    nx.draw(Gref3, posref3, with_labels=True, node_color=colors, edge_color='gray',
            node_size=500, font_color='white', font_weight='bold', ax=ax)
    ax.set_title(title, fontsize=10)
plt.suptitle('Reference Graph N=21, M=35, seed=30', fontsize=12)
plt.tight_layout()
plt.show()

print(pd.DataFrame(kl_h3).to_string(index=False))

---
## Summary

| Algorithm | Gain metric | Swaps per pass | Complexity |
|-----------|-------------|----------------|------------|
| Kernighan-Lin | $D(a) + D(b) - 2c_{ab}$ | Up to $n/2$ (best prefix) | $\mathcal{O}(n^2 \log n)$ |
| KL Variant | Degree-normalised $dd(i,j)$ | 1 (greedy) | $\mathcal{O}(n^2)$ per pass |

Both algorithms enforce balanced partitions ($|A| = |B|$) and terminate at a local optimum.